# CME Futures: Risk Overlays

For each return horizon, this notebook selects the highest validation Sharpe from the immutable
union of signal and allocation results, then applies every position-level risk rule declared in
the case-study configuration. Stop-loss, trailing-stop, and time-exit parameters are fixed before
the validation backtest. They are not calibrated from the same validation price path they assess.

Risk rules execute inside the existing futures engine after product-keyed target decisions cross
the typed boundary. Every declared rule must finish, and the resulting per-label candidate sets
remain eligible for final validation selection.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# CME 期貨：風險覆蓋層

對每個報酬率預測期間，本 Notebook 會從不可變的訊號與配置結果聯集中，選出驗證期夏普比率最高者，再套用案例研究設定中聲明的每項部位層級風險規則。停損、移動停損與時間出場參數會在驗證期回測前固定，不會根據同一段受評估的驗證期價格路徑校準。

商品鍵值的目標決策通過型別化邊界後，風險規則會在既有期貨引擎內執行。每項聲明的規則都必須執行完畢；產生的依標籤區分候選集合，仍可參與最終驗證期選擇。
<!-- ML4T_ZH_END c0000-p000 -->


## What a risk overlay is, and why it is a separate stage

The stages before this one decided *what to hold*: a signal ranked the products, an allocation
rule decided how much of each. A risk overlay decides *when to stop holding it* - it sits on
top of an existing set of positions and closes them on a condition the signal never
considered.

The three rules here are the standard family. A **stop-loss** exits when a position has lost
more than a set amount from entry. A **trailing stop** exits when it has given back a set
amount from its best level, so it protects an unrealized gain rather than only the entry
price. A **time exit** closes after a fixed holding period whatever the position is doing, on
the reasoning that a signal with a horizon has nothing to say beyond it.

### The asymmetry these introduce, which is the point and the danger

A signal is symmetric about its own prediction: it is as willing to be wrong in one direction
as the other. A stop is not. It truncates the loss side of the distribution and leaves the
gain side alone, and that is why it appeals.

What it also does is convert an unrealized loss into a realized one at the worst available
moment, and give up any recovery that would have followed. For a mean-reverting signal - which
describes carry, the signal this case study trades - that is a direct conflict: the position
is exited precisely when the thing the signal is betting on has become most attractive. A stop
on a mean-reverting strategy is not a free reduction in risk. It is a change to the strategy,
and it can easily be a change for the worse.


<!-- ML4T_ZH_BEGIN c0001-p000 -->
## 風險覆蓋層是什麼，為何要獨立成一個階段

前面幾個階段決定*持有什麼*：訊號為商品排序，配置規則決定各自持有多少。風險覆蓋層則決定*何時停止持有*；它加在既有部位之上，並在訊號從未考慮的條件成立時平倉。

此處的三種規則是標準類型。**停損**會在部位相對進場時虧損超過設定幅度後出場。**移動停損**會在部位從最佳價位回落超過設定幅度後出場，因此保護的是未實現收益，而不只是進場價。**時間出場**則在固定持有期間後平倉，不論部位表現如何；理由是有特定預測期間的訊號，對超出該期間的情況沒有判斷。

### 這些規則帶來的不對稱性：既是目的，也是風險

訊號對自身的預測具有對稱性：它可能在任一方向上同樣出錯。停損則不同。它截短報酬分布的虧損側，保留獲利側，這正是它吸引人的地方。

但停損也會在最不利的可用時點，將未實現虧損轉成已實現虧損，並放棄之後可能出現的反彈。對均值回歸訊號而言——本案例交易的持有收益訊號即屬此類——這是直接衝突：部位恰好在訊號所押注的機會變得最有吸引力時出場。對均值回歸策略加上停損，並非毫無代價地降低風險，而是改變策略，而且很容易讓策略變差。
<!-- ML4T_ZH_END c0001-p000 -->

That is the whole reason this is measured rather than assumed. Risk management is the part of
a strategy where intuition is least reliable and where "obviously prudent" is applied without
testing more often than anywhere else in the pipeline.

### Why the parameters are fixed before the backtest, and not after

The stop distances and holding periods come from `config/setup.yaml` and are fixed before the
validation backtest runs. They are deliberately **not** calibrated on the price path they are
then assessed against.

The reason is that this stage is unusually easy to cheat at without noticing. Choosing a stop
level by trying several and keeping the one with the best validation Sharpe would find the
level that best avoided the particular drawdowns that particular history happened to contain,
and would report the result as a risk improvement. Nothing about it would generalize, and
nothing in the output frame would show what happened - the returns would simply look better.
Fixing the parameters in configuration is what makes the comparison between overlay and no
overlay a real one.

### Why every declared rule must finish

A rule that failed and was skipped would leave a candidate set that silently means "the rules
that happened to work", and the selection downstream would then choose from it as though it
were the declared set. Failing the notebook is the only outcome that keeps the population
equal to the configuration.

### These candidates stay eligible


<!-- ML4T_ZH_BEGIN c0001-p001 -->
這正是必須實際衡量，而不能直接假定效果的原因。在策略各環節中，風險管理最不適合只憑直覺判斷；「顯然審慎」的做法也最常在此未經測試就被採用。

### 為何參數在回測前，而非回測後固定

停損距離與持有期間取自 `config/setup.yaml`，在驗證期回測執行前即已固定。它們刻意**不會**根據後續用來評估的價格路徑校準。

原因是這個階段特別容易在不自覺間作弊。若嘗試數個停損水準，再選擇驗證期夏普比率最高者，就會找出最能避開該段歷史中特定回撤的水準，卻將結果報告為風險改善。這種效果無法推廣，輸出資料表也不會顯示發生了什麼；報酬率看起來只會比較好。把參數固定在設定中，才能讓有無覆蓋層之間的比較真正成立。

### 為何每項聲明的規則都必須執行完畢

若某項規則失敗後遭到略過，候選集合的實際意義就悄悄變成「碰巧能運作的規則」，後續選擇卻仍會把它當成原本聲明的集合。讓 Notebook 失敗，是唯一能確保群體與設定一致的做法。

### 這些候選結果仍有入選資格
<!-- ML4T_ZH_END c0001-p001 -->

Unlike the cost sweep, risk-overlay results are part of the final selection pool.
`19_strategy_analysis` selects over the union of signal, allocation and risk-overlay
backtests, so an overlay that genuinely improves validation Sharpe can be what the case study
ships - and one that does not is visible as such next to the configuration it was applied to.

<!-- ML4T_ZH_BEGIN c0001-p002 -->
與成本掃描不同，風險覆蓋層的結果是最終選擇集合的一部分。
`19_strategy_analysis` 會從訊號、配置與風險覆蓋層回測的聯集中選擇，因此真正改善驗證期夏普比率的覆蓋層，可能成為本案例研究最後採用的方法；未改善者也會與其所套用的設定並列顯示。
<!-- ML4T_ZH_END c0001-p002 -->


In [1]:
"""Run the declared CME futures risk-overlay population."""

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    create_label_candidate_sets,
    open_study,
    pre_overlay_results,
    product_universe_table,
    rank_by_validation_sharpe,
    run_official_backtest_requests,
    strategy_request_frame,
)
from case_studies.research.population import supersedes_for_run
from case_studies.utils.sweep_config import get_position_risk_controls, get_top_n_predictions

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_LABELS: list[str] = []

# The risk population is immutable under its name, so a run whose members have moved has to say
# which generation it retires. Anything upstream that changes a backtest identity moves them - a
# corrected label, a changed accounting field, a re-run after a registry reset - and
# `OfficialPopulation.create` refuses to write a different member list under a name that already
# exists. Declared as a literal so that running the committed notebook as it stands recomputes
# the population on record. Empty for a first snapshot.
RISK_POPULATION = "cme_futures-risk-validation-v1"
SUPERSEDES_RISK_POPULATION: str = ""

# How many parents per label the overlay grid sits on. `None` reads
# `backtest.sweep.top_n_predictions.risk_overlay`, which every case study declares as 1, and one
# is narrow on purpose: an overlay is a second search over the same validation folds, so the
# question the book asks is whether a control improves the configuration the funnel already
# chose. Until 2026-09-20 that 1 was a literal `[0]` below rather than a number read from the
# declaration, which left this case study unable to answer at any other width while four others
# could. A run at a wider width changes the member list of every name this notebook publishes,
# so it needs its own `RISK_POPULATION` and `SUPERSEDES_CANDIDATE_SETS` the same way a narrowed
# run does.
TOP_N_COMBOS = None

# The per-label candidate sets this notebook freezes are immutable under their names too, and
# for the same reason as the population above: `CandidateSet.create` refuses a changed member
# list under a name that already exists. Nothing reached that argument before, so any run whose
# membership moved - which a wider sweep does by construction - stopped at the freeze after the
# fit, with no parameter able to answer it.
#
# Each name maps to the generation this run retires. `"live"` names the lineage and looks the
# generation up, which is the form that does not decay: naming the head instead is correct only
# until the next publish, because `create` accepts the head and nothing else. The declaration is
# resolved through `candidate_set_supersedes` rather than offered straight, so a reader's clean
# clone - which has no generation to replace, and often no `candidate_sets` table at all -
# publishes generation one instead of being refused. An unchanged re-run never reads it: a set's
# hash is computed from its members and its contract, so the existing name binding answers.
SUPERSEDES_CANDIDATE_SETS: dict[str, str] = {
    "cme_futures-risk-fwd_ret_5d-v1": "live",
    "cme_futures-risk-fwd_ret_21d-v1": "live",
}

## Fixed per-label inputs and risk rules

No candidate cap or runtime-dependent skip is allowed. The configured list is the population.

**What the overlay is applied to.** An overlay needs an existing strategy to sit on, and there
is one per label: the highest validation Sharpe from the immutable union of the signal and
allocation stages. Taking the best of the two stages rather than the signal alone matters,
because an overlay applied to a weaker parent would be measuring the overlay against a
strategy the case study would not have shipped anyway.

**Why per label rather than one overall.** Each return horizon is a different prediction
problem and its best configuration is chosen within its own horizon. Picking one parent across
all labels would let the strongest horizon's configuration stand in for horizons it was never
fitted for, and the overlay comparison would then be confounded by which label the parent came
from. Every configured rule runs against every label's own parent, so the comparison within a
label is like for like.

**Why the configured list is the population, with no cap.** A runtime cap would make the set
depend on how long the run took, which means a re-run could select from a different set and
nothing would record that it had. The rules are declared in configuration precisely so the
population is a property of the configuration rather than of the execution.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 固定的依標籤區分輸入與風險規則

不允許設定候選數量上限，也不允許依執行時情況略過項目。設定中列出的規則就是完整群體。

**覆蓋層套用到什麼。** 覆蓋層必須附加在既有策略上，而每個標籤各有一個：從不可變的訊號與配置階段聯集中，選出驗證期夏普比率最高者。取兩個階段中的最佳者，而非只取訊號階段最佳者，十分重要；若將覆蓋層套在較弱的父策略上，衡量的便是它相對於本案例研究原本就不會採用的策略之表現。

**為何依標籤分別處理，而非選一個整體最佳者。** 每個報酬率預測期間都是不同的預測問題，最佳設定應在各自的期間內選出。若跨所有標籤選一個父策略，就會讓最強期間的設定代替它從未針對其擬合的其他期間，覆蓋層比較也會受到父策略來源標籤的干擾。每項設定規則都套用到每個標籤自己的父策略，使標籤內的比較基礎一致。

**為何設定清單就是完整群體，且不設上限。** 執行時上限會讓集合取決於執行耗時，重新執行時便可能從不同集合中選擇，卻沒有任何紀錄。規則明定於設定中，正是為了讓群體由設定決定，而非由執行過程決定。
<!-- ML4T_ZH_END c0004-p000 -->


In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
if EXECUTION_TIER == "canonical":
    if PREVIEW_LABELS:
        raise ValueError("canonical execution cannot declare preview reductions")
    labels = ALL_LABELS
elif EXECUTION_TIER == "preview":
    if WORKSPACE is None or not PREVIEW_LABELS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_LABELS")
    unknown = sorted(set(PREVIEW_LABELS) - set(ALL_LABELS))
    if unknown:
        raise ValueError(f"preview labels this case study does not declare: {unknown}")
    labels = tuple(PREVIEW_LABELS)
else:
    raise ValueError(f"unsupported execution tier: {EXECUTION_TIER!r}")
if TOP_N_COMBOS is None:
    TOP_N_COMBOS = get_top_n_predictions("cme_futures", "risk_overlay")
if TOP_N_COMBOS < 1:
    raise ValueError("the risk overlay needs at least one parent per label")
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [4]:
risk_controls = get_position_risk_controls("cme_futures")
if not risk_controls:
    raise ValueError("the configured position-risk population is empty")

request_rows = []
for label in labels:
    ranked = rank_by_validation_sharpe(
        study,
        pre_overlay_results(
            study,
            label=label,
            execution_tier=EXECUTION_TIER,
            supersedes_by_set=SUPERSEDES_CANDIDATE_SETS,
        ),
    )
    for selected in ranked[:TOP_N_COMBOS]:
        strategy = selected.spec()["strategy"]
        prediction_hash = selected.registry_record()["prediction_hash"]
        for control in risk_controls:
            rule = {key: value for key, value in control.items() if key != "name"}
            request_rows.append(
                {
                    "request_name": f"{selected.hash}-risk-{control['name']}",
                    "prediction_hash": prediction_hash,
                    "label": label,
                    "signal": strategy["signal"],
                    "allocation": strategy.get("allocation"),
                    "risk": {"position_rules": [rule]},
                    "costs": None,
                    "chapter": "ch19",
                }
            )
requests = strategy_request_frame(request_rows)
requests.select("request_name", "prediction_hash", "label", "risk")

request_name,prediction_hash,label,risk
str,str,str,object
"""5ff8aaf3a937-risk-stop_loss_3p…","""b467971cd825""","""fwd_ret_5d""","{'position_rules': [{'type': 'stop_loss', 'threshold': 0.03}]}"
"""5ff8aaf3a937-risk-stop_loss_5p…","""b467971cd825""","""fwd_ret_5d""","{'position_rules': [{'type': 'stop_loss', 'threshold': 0.05}]}"
"""5ff8aaf3a937-risk-stop_loss_10…","""b467971cd825""","""fwd_ret_5d""","{'position_rules': [{'type': 'stop_loss', 'threshold': 0.1}]}"
"""5ff8aaf3a937-risk-stop_loss_15…","""b467971cd825""","""fwd_ret_5d""","{'position_rules': [{'type': 'stop_loss', 'threshold': 0.15}]}"
"""5ff8aaf3a937-risk-trailing_1pc…","""b467971cd825""","""fwd_ret_5d""","{'position_rules': [{'type': 'trailing_stop', 'threshold': 0.01}]}"
…,…,…,…
"""48a93aa5d08d-risk-trailing_15p…","""430e5789167d""","""fwd_ret_21d""","{'position_rules': [{'type': 'trailing_stop', 'threshold': 0.15}]}"
"""48a93aa5d08d-risk-trailing_20p…","""430e5789167d""","""fwd_ret_21d""","{'position_rules': [{'type': 'trailing_stop', 'threshold': 0.2}]}"
"""48a93aa5d08d-risk-time_exit_10""","""430e5789167d""","""fwd_ret_21d""","{'position_rules': [{'type': 'time_exit', 'bars': 10}]}"


## Execute and freeze risk candidates

Each request carries the fitted prediction checkpoint, product decisions, fold-transition policy,
contract and roll inputs, and one risk rule. Missing members fail before the candidate set exists.

One request is one rule applied to one parent, so a rule's effect is read against its own
parent rather than against the field. Two rules that both improve Sharpe are not therefore
combinable: they may exit on the same moves, and their joint effect is not the sum of their
separate ones. Nothing here estimates that, and a reader stacking rules on the strength of
this table would be assuming an additivity it does not measure.

The results are frozen as a named population, and `SUPERSEDES_RISK_POPULATION` in the
parameter cell is how a re-run names the generation it retires. A retired snapshot stays in
the registry rather than being deleted, so a Sharpe quoted from an earlier generation remains
traceable to the population it was computed over.

<!-- ML4T_ZH_BEGIN c0007-p000 -->
## 執行並封存風險候選結果

每個請求都包含已擬合的預測檢查點、商品決策、折間狀態轉換政策、合約與轉倉輸入，以及一項風險規則。若有成員缺失，會在候選集合建立前失敗。

一個請求代表一項規則套用到一個父策略，因此規則的效果應與其自身父策略比較，而非與所有候選結果比較。即使兩項規則都改善夏普比率，也不表示可以合併使用：它們可能在相同波動時出場，合併效果不會等於各自效果之和。此處沒有估計合併效果；若讀者根據這張表疊加規則，就等於假設了未經衡量的可加性。

結果會封存為具名群體；參數儲存格中的 `SUPERSEDES_RISK_POPULATION`，是重新執行時指定要汰換哪個版本的方式。已汰換的快照仍留在登錄資料庫，不會刪除，因此引用舊版本的夏普比率時，仍可追溯到其計算所依據的群體。
<!-- ML4T_ZH_END c0007-p000 -->


In [5]:
execution = run_official_backtest_requests(
    study,
    requests,
    population_name=RISK_POPULATION if EXECUTION_TIER == "canonical" else None,
    supersedes=supersedes_for_run(
        study,
        population_name=RISK_POPULATION,
        declared=SUPERSEDES_RISK_POPULATION or None,
        execution_tier=EXECUTION_TIER,
    ),
)
candidate_sets = (
    create_label_candidate_sets(
        study, execution, stage="risk", supersedes_by_set=SUPERSEDES_CANDIDATE_SETS
    )
    if EXECUTION_TIER == "canonical"
    else {}
)

`source` says whether each member was computed by this run or served from the registry because
an identical identity was already recorded. A re-run of a registered sweep is entirely `reused`
and completes in seconds; without the column that is indistinguishable from having computed
every row.

<!-- ML4T_ZH_BEGIN c0009-p000 -->
`source` 表示每個成員是本次執行新計算的，還是因為相同識別資訊已記錄在登錄資料庫，而直接取用既有結果。重新執行已登錄的掃描時，所有成員都會是 `reused`，幾秒內即可完成；若沒有這一欄，就無法區分它與重新計算了每一列的情況。
<!-- ML4T_ZH_END c0009-p000 -->


In [6]:
execution.catalog_rows.sort("label", "request_name")

request_name,label,prediction_hash,decision_hash,backtest_hash,complete,source
str,str,str,str,str,bool,str
"""48a93aa5d08d-risk-stop_loss_10…","""fwd_ret_21d""","""430e5789167d""","""d0b23052f09f""","""6060c740ed89""",true,"""computed"""
"""48a93aa5d08d-risk-stop_loss_15…","""fwd_ret_21d""","""430e5789167d""","""22e1d1a158cd""","""db9499e53ab8""",true,"""computed"""
"""48a93aa5d08d-risk-stop_loss_3p…","""fwd_ret_21d""","""430e5789167d""","""1c24abc45138""","""532b054de36b""",true,"""computed"""
"""48a93aa5d08d-risk-stop_loss_5p…","""fwd_ret_21d""","""430e5789167d""","""f59ec353070c""","""62b3fd0cddc2""",true,"""computed"""
"""48a93aa5d08d-risk-time_exit_10""","""fwd_ret_21d""","""430e5789167d""","""aa06696c15d7""","""d6032351e0e1""",true,"""computed"""
…,…,…,…,…,…,…
"""5ff8aaf3a937-risk-trailing_1pc…","""fwd_ret_5d""","""b467971cd825""","""66652cbe174c""","""b1f5e1e25451""",true,"""computed"""
"""5ff8aaf3a937-risk-trailing_20p…","""fwd_ret_5d""","""b467971cd825""","""599ca8b90cfe""","""0d3740e1b5a6""",true,"""computed"""
"""5ff8aaf3a937-risk-trailing_2pc…","""fwd_ret_5d""","""b467971cd825""","""0c9fc1f8f910""","""862591b426bc""",true,"""computed"""


Final selection in `19_strategy_analysis` uses the union of signal, allocation, and risk-overlay
results. Cost-sensitivity rows are excluded.

One consequence to carry into `16_costs`: an overlay only ever adds trades. Every stop that
fires is an exit that the signal did not ask for, and often a re-entry afterwards. So an
overlay that improves Sharpe here can still be the worse strategy once friction is priced, and
the two notebooks have to be read together rather than in sequence. This is also why the
selected configuration is priced with its overlay in place rather than bare.

<!-- ML4T_ZH_BEGIN c0011-p000 -->
`19_strategy_analysis` 的最終選擇會使用訊號、配置與風險覆蓋層結果的聯集，排除成本敏感度資料列。

進入 `16_costs` 時須記住一項影響：覆蓋層只會增加交易。每次觸發停損，都是訊號未要求的出場，之後往往還要重新進場。因此，即使覆蓋層在此改善了夏普比率，將交易摩擦成本計入後仍可能是較差的策略；兩份 Notebook 必須合併解讀，而不是各自孤立解讀。這也是為何所選設定在計價時會保留覆蓋層，而非只計價未加覆蓋層的策略。
<!-- ML4T_ZH_END c0011-p000 -->
